<a href="https://colab.research.google.com/github/MuhammadJawadFasih/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadJawadFasih/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
if IN_COLAB:
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Ready.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

This queue ranks every page by predicted decline probability from the w05_model logistic regression, trained on the client-holdout split. Each page gets one of three reason codes based on which signals are driving its risk, so a human reviewer sees a plain-language reason, not just a number.

Reason codes:
- STALE_AND_FADING: low recent traffic (users_90d, sessions_90d below median) combined with a long time since last update.
- VISIBLE_BUT_DECLINING: still earns meaningful impressions, but predicted probability is high — a page worth reviewing precisely because it hasn't disappeared yet.
- LOW_SIGNAL: low predicted probability and low traffic across the board — deprioritize, likely not worth an editor's time either way.

Action label for all flagged pages: REVIEW_REFRESH (decision-support only).

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

feature_columns = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]

model_df = df[["content_id", "client_id", "is_declining_label"] + feature_columns].copy()
model_df[feature_columns] = model_df[feature_columns].replace([np.inf, -np.inf], np.nan)
for col in feature_columns:
    model_df[col] = model_df[col].fillna(model_df[col].median())

# Train on client-holdout split (same as w05_model), then score everyone
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(model_df, groups=model_df["client_id"]))
train = model_df.iloc[train_idx]

model = Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=2000, random_state=42))])
model.fit(train[feature_columns], train["is_declining_label"])

model_df["predicted_probability"] = model.predict_proba(model_df[feature_columns])[:, 1]

# Reason codes
users_median = model_df["users_90d"].median()
sessions_median = model_df["sessions_90d"].median()

def assign_reason(row):
    stale = row["days_since_last_update"] >= 180
    low_traffic = row["users_90d"] < users_median and row["sessions_90d"] < sessions_median
    if row["predicted_probability"] >= 0.5 and stale and low_traffic:
        return "STALE_AND_FADING"
    elif row["predicted_probability"] >= 0.5:
        return "VISIBLE_BUT_DECLINING"
    else:
        return "LOW_SIGNAL"

model_df["reason_code"] = model_df.apply(assign_reason, axis=1)
model_df["action"] = np.where(model_df["predicted_probability"] >= 0.5, "REVIEW_REFRESH", "NO_ACTION")

queue = model_df.sort_values("predicted_probability", ascending=False).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

print(f"Total pages ranked: {len(queue)}")
print(f"Flagged for review: {(queue['action'] == 'REVIEW_REFRESH').sum()}")
print("\nReason code breakdown:")
print(queue["reason_code"].value_counts())
print("\nTop 10:")
print(queue[["rank", "content_id", "predicted_probability", "action", "reason_code"]].head(10).to_string(index=False))

Total pages ranked: 30000
Flagged for review: 17736

Reason code breakdown:
reason_code
VISIBLE_BUT_DECLINING    17644
LOW_SIGNAL               12264
STALE_AND_FADING            92
Name: count, dtype: int64

Top 10:
 rank           content_id  predicted_probability         action           reason_code
    1 content_4560b0a818ab               1.000000 REVIEW_REFRESH VISIBLE_BUT_DECLINING
    2 content_8e7ba84a972b               0.999779 REVIEW_REFRESH VISIBLE_BUT_DECLINING
    3 content_c8ad1f4d0e56               0.994737 REVIEW_REFRESH VISIBLE_BUT_DECLINING
    4 content_a22b7f6c73c5               0.994388 REVIEW_REFRESH VISIBLE_BUT_DECLINING
    5 content_70b8f5323e29               0.983860 REVIEW_REFRESH VISIBLE_BUT_DECLINING
    6 content_c53566c0e1b1               0.974045 REVIEW_REFRESH VISIBLE_BUT_DECLINING
    7 content_69fad7e6c50c               0.964118 REVIEW_REFRESH VISIBLE_BUT_DECLINING
    8 content_3f165aff4181               0.955384 REVIEW_REFRESH VISIBLE_BUT_DECLINING
 

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended user: a content strategist or SEO lead deciding which pages to prioritize for review out of a large portfolio, using this as a starting shortlist rather than a final verdict.

Where it stops being valid:
- This model was trained on a 30,000-row anonymized sample from one snapshot in time; it should not be assumed to generalize to a different content portfolio, industry, or time period without re-validation.
- The label (trend_direction == "down") reflects a 30-day-vs-prior-30-day impression change — it says a page's search visibility recently dropped, not that the content itself is low quality, nor that a refresh will fix it.
- Precision@50 was 0.720 on a held-out set of only 7 clients — a small evaluation sample, so the true real-world precision could differ meaningfully.
- This is an observed, decision-support ranking only. It is not a causal claim about what refreshing a page will do.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Intended use: shortlist generator for editorial review, not an automated action system.")
print("Validity boundary: single anonymized snapshot, 7-client holdout, no causal claims.")
print(f"Evaluation Precision@50 (from w05_model, honest split): 0.720")

Intended use: shortlist generator for editorial review, not an automated action system.
Validity boundary: single anonymized snapshot, 7-client holdout, no causal claims.
Evaluation Precision@50 (from w05_model, honest split): 0.720


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on any flagged page, a human should check:
- Does the page still serve a real business purpose (is it still relevant to current offerings/topics)?
- Is the "decline" explained by something outside content quality — a seasonal query, a SERP feature change, or a competitor entering the space?
- Does the page have zero CTR because of intent mismatch (e.g., informational queries where users don't need to click) rather than a fixable content problem?

Never automate:
- Auto-publishing or auto-editing content changes based on this score alone.
- Removing or deprioritizing pages without a human confirming the page's business relevance first.
- Treating the ranked list as a performance review of the content's original author.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("No-go list: no auto-editing, no auto-removal, no automated authorship judgments.")
print("Every REVIEW_REFRESH page requires human confirmation of business relevance before action.")

No-go list: no auto-editing, no auto-removal, no automated authorship judgments.
Every REVIEW_REFRESH page requires human confirmation of business relevance before action.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Retrain or re-audit the model if any of the following happen:
- The portfolio's overall decline base rate shifts meaningfully from the current 0.542 (e.g., beyond +/-0.10) — the label distribution the model was trained on no longer matches reality.
- Precision@50 on a fresh holdout drops materially below the current 0.720 benchmark when re-evaluated on new data.
- A new content type, intent category, or traffic source (e.g., a new AI referral channel) appears that wasn't represented in the training data.
- More than ~6 months pass since the last training snapshot, since search behavior and algorithm changes can shift the underlying patterns.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
current_base_rate = df["is_declining_label"].mean()
print(f"Current decline base rate (retrain trigger reference point): {current_base_rate:.3f}")
print("Retrain if: base rate drifts >0.10 from this value, OR fresh-holdout Precision@50 drops below ~0.65,")
print("OR >6 months pass since this training snapshot.")

Current decline base rate (retrain trigger reference point): 0.542
Retrain if: base rate drifts >0.10 from this value, OR fresh-holdout Precision@50 drops below ~0.65,
OR >6 months pass since this training snapshot.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Writing the final ranked queue to work/outputs/ for reuse in the capstone paper's Results and Recommendations sections.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

output_cols = ["rank", "content_id", "predicted_probability", "action", "reason_code",
               "days_since_last_update", "impressions_90d", "avg_position", "ctr"]
output_path = Path("work/outputs/action_playbook_queue.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
queue[output_cols].to_csv(output_path, index=False)

print("Exported:", output_path)
print("Rows:", len(queue))

Exported: work/outputs/action_playbook_queue.csv
Rows: 30000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.